# Notebook 1 — Qwen 3.5 9B: FullDist + Ranked Nodes guiado por confiança — Google Colab

**Objetivo:** gerar CPTs sem consultar a referência `bnRep`, usando uma estratégia híbrida para reduzir chamadas ao LLM.

Esta versão foi preparada para **Google Colab com GPU** e executa quatro condições independentes:

1. controle sem persona;
2. engenheiro estrutural;
3. sismologista;
4. analista de risco.

O protocolo remove o `SepState`: usa **FullDist nas linhas em que o LLM declara confiança suficiente** e **Ranked Nodes nas demais linhas elegíveis**.

**Esta versão não monta nem solicita acesso ao Google Drive.**  
Você envia manualmente os dois JSONs para o ambiente do Colab e os resultados ficam em `/content/`.


## 0. Preparar o Google Colab

Antes de executar, selecione **Ambiente de execução > Alterar tipo de ambiente de execução > GPU**.

A célula seguinte instala as dependências necessárias. Em GPUs comuns do Colab, como T4 ou L4, o notebook ativa **NF4 4 bits automaticamente** para caber na VRAM. Em GPUs com mais memória, como A100, ele mantém BF16/FP16 por padrão.

> Para comparação científica direta com uma execução anterior, mantenha a mesma precisão/quantização entre os modelos ou registre essa diferença como parte da configuração experimental.

In [ ]:
# ============================================================
# INSTALAÇÃO DAS DEPENDÊNCIAS NO GOOGLE COLAB
# Execute esta célula antes das demais.
# ============================================================
import sys, subprocess, importlib.util, os
from pathlib import Path
os.environ.setdefault('HF_HOME','/content/hf_cache')
Path(os.environ['HF_HOME']).mkdir(parents=True, exist_ok=True)

pacotes = [
    "git+https://github.com/huggingface/transformers.git",
    "accelerate>=1.0.0",
    "bitsandbytes>=0.46.1",
    "huggingface_hub>=0.25.0",
    "pgmpy>=1.0.0",
    "scipy>=1.11.0",
    "networkx>=3.2",
    "pandas>=2.0.0",
]

print("[COLAB] Instalando/atualizando dependências...")
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q", "-U", *pacotes
])

necessarios = (
    "torch", "transformers", "accelerate", "bitsandbytes",
    "huggingface_hub", "pgmpy", "numpy", "networkx", "scipy", "pandas"
)
faltantes = [n for n in necessarios if importlib.util.find_spec(n) is None]
if faltantes:
    raise RuntimeError("Bibliotecas faltantes após a instalação: " + ", ".join(faltantes))

import transformers
print("[DEPENDÊNCIAS] OK | transformers =", transformers.__version__)

## 0.1. Arquivos de entrada — sem Google Drive

Envie manualmente, pelo painel **Arquivos** do Google Colab, os dois arquivos:

- JSON da Rede Bayesiana;
- `personas.json`.

Por padrão, o notebook procura:

- `/content/rede.json`
- `/content/personas.json`

Se os seus arquivos tiverem outros nomes, altere apenas os dois caminhos na célula abaixo.

**Nenhum acesso ao Google Drive é solicitado.**


In [ ]:
# ============================================================
# ARQUIVOS LOCAIS DO COLAB — SEM GOOGLE DRIVE
# ============================================================
from pathlib import Path
import os

# 1) Faça upload manual dos arquivos pelo painel "Arquivos" do Colab.
# 2) Ajuste somente estes caminhos se os nomes forem diferentes.
CAMINHO_REDE_MANUAL = Path('/content/rede.json')
CAMINHO_PERSONAS_MANUAL = Path('/content/personas.json')

if not CAMINHO_REDE_MANUAL.is_file():
    raise FileNotFoundError(
        f'JSON da rede não encontrado em {CAMINHO_REDE_MANUAL}. '
        'Envie o arquivo manualmente para /content ou ajuste CAMINHO_REDE_MANUAL.'
    )

if not CAMINHO_PERSONAS_MANUAL.is_file():
    raise FileNotFoundError(
        f'JSON de personas não encontrado em {CAMINHO_PERSONAS_MANUAL}. '
        'Envie o arquivo manualmente para /content ou ajuste CAMINHO_PERSONAS_MANUAL.'
    )

# Tudo fica local no runtime do Colab.
PASTA_PROJETO = Path('/content/Qwen_BN_Confidence_RN')
PASTA_PROJETO.mkdir(parents=True, exist_ok=True)

os.environ['EXPERIMENT_BASE_DIR'] = str(PASTA_PROJETO)
os.environ['REDE_JSON'] = str(CAMINHO_REDE_MANUAL)
os.environ['PERSONAS_JSON'] = str(CAMINHO_PERSONAS_MANUAL)
os.environ['HF_HOME'] = '/content/hf_cache'
Path(os.environ['HF_HOME']).mkdir(parents=True, exist_ok=True)

print('Rede:', CAMINHO_REDE_MANUAL)
print('Personas:', CAMINHO_PERSONAS_MANUAL)
print('Projeto/saídas:', PASTA_PROJETO)
print('Cache do modelo:', os.environ['HF_HOME'])


In [ ]:
# ============================================================
# CONFIGURAÇÃO PRINCIPAL — GOOGLE COLAB
# ============================================================
import os, json, time, re, itertools, hashlib, random, platform
from pathlib import Path
from typing import List, Dict, Any, Optional, Tuple
from datetime import datetime, timezone
import torch

COLAB_MODE = True
BASE_DIR = Path(os.environ.get('EXPERIMENT_BASE_DIR', '/content/Qwen_BN_Confidence_RN'))
BASE_DIR.mkdir(parents=True, exist_ok=True)

CAMINHO_JSON = Path(os.environ.get('REDE_JSON', str(BASE_DIR/'entradas'/'rede.json')))
CAMINHO_PERSONAS = Path(os.environ.get('PERSONAS_JSON', str(BASE_DIR/'entradas'/'personas.json')))

if not torch.cuda.is_available():
    raise RuntimeError(
        'GPU CUDA não encontrada. No Colab, escolha: Ambiente de execução > '
        'Alterar tipo de ambiente de execução > GPU.'
    )

_gpu = torch.cuda.get_device_properties(0)
GPU_VRAM_GB = _gpu.total_memory / 2**30
print(f'[GPU] {_gpu.name} | VRAM: {GPU_VRAM_GB:.1f} GiB')

HF_MODEL_ID = 'Qwen/Qwen3.5-9B'
QWEN35_SEM_THINKING = True
HF_MODEL_REVISION = os.environ.get('HF_MODEL_REVISION', 'main')
PERGUNTAR_FAMILIARIDADE_REDE = os.environ.get('PERGUNTAR_FAMILIARIDADE_REDE','1').lower() not in ('0','false','no')

NOME_REDE_BNREP = os.environ.get('BNREP_REDE', 'seismic')
if CAMINHO_JSON.is_file() and not os.environ.get('BNREP_REDE'):
    _metadados_rede = json.loads(CAMINHO_JSON.read_text(encoding='utf-8-sig'))
    _rede_info = _metadados_rede.get('network', {})
    if isinstance(_rede_info, dict):
        NOME_REDE_BNREP = _rede_info.get('bnrep_name', _rede_info.get('bnrep_id', NOME_REDE_BNREP))
if not isinstance(NOME_REDE_BNREP,str) or not re.fullmatch(r'[A-Za-z0-9_.-]{1,100}',NOME_REDE_BNREP):
    raise ValueError('Nome bnRep inválido.')

EXECUTAR_EXPERIMENTO = True
EXECUTAR_GERACAO = True
SEED = 42
TEMPERATURE = 0.1
MAX_RETRIES = 5

# "auto": usa NF4 em GPUs com menos de 32 GiB; BF16/FP16 nas maiores.
# Para forçar: use "4bit" ou "bf16".
MODO_QUANTIZACAO = os.environ.get('QWEN_QUANTIZACAO', 'auto').strip().lower()
if MODO_QUANTIZACAO not in ('auto','4bit','bf16','fp16','none'):
    raise ValueError('QWEN_QUANTIZACAO deve ser auto, 4bit, bf16, fp16 ou none.')

if MODO_QUANTIZACAO == 'auto':
    USAR_4BITS = GPU_VRAM_GB < 32
elif MODO_QUANTIZACAO == '4bit':
    USAR_4BITS = True
else:
    USAR_4BITS = False

if USAR_4BITS:
    print('[QUANTIZAÇÃO] NF4 4 bits ativado para reduzir uso de VRAM.')
    print('[ATENÇÃO METODOLÓGICA] Se sua execução anterior usou BF16/FP16, registre a mudança de precisão.')
else:
    print('[QUANTIZAÇÃO] Desativada: execução em BF16/FP16 conforme a GPU.')

MAX_NEW_TOKENS_FULLDIST = 2048
MAX_NEW_TOKENS_FULLDIST_RETRY = 4096
MAX_NEW_TOKENS_CONFIANCA = 1536
MAX_RETRIES_CONFIANCA = 4

LIMIAR_CONFIANCA = float(os.environ.get('LIMIAR_CONFIANCA','0.80'))
if not 0 <= LIMIAR_CONFIANCA <= 1:
    raise ValueError('LIMIAR_CONFIANCA precisa estar entre 0 e 1.')

RN_SIGMA_DEFAULT = 0.16
RN_SIGMA_MIN = 0.03
RN_SIGMA_MAX = 0.45
RN_REGULARIZACAO_PESOS = 0.02
RN_ENUMERAR_DIRECOES_ATE = 6
RN_EPS = 1e-12

ASSUMIR_ORDEM_ESTADOS_SEM_METADADO = False

REPETICOES = 1
SOMENTE_RECOMBINAR = False
EXIGIR_TRES_PERSONAS = True
METODO_HIBRIDO = 'ConfidenceRankedNodes'

ID_SESSAO = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
RESULTADOS = BASE_DIR/'resultados_qwen35_9b_confidence_rn'
RESULTADOS.mkdir(parents=True,exist_ok=True)
ID_JSON = hashlib.sha256(CAMINHO_JSON.read_bytes()).hexdigest()[:12] if CAMINHO_JSON.is_file() else 'sem_json'
MODEL_SLUG = re.sub(r'[^A-Za-z0-9_.-]+','_',HF_MODEL_ID)[:110]
DIR_ESTUDO = RESULTADOS/f'{NOME_REDE_BNREP}_{ID_JSON}'/MODEL_SLUG
DIR_ESTUDO.mkdir(parents=True,exist_ok=True)
CAMINHO_BIF_GOLD = None
REALIZAR_INFERENCIA = EXECUTAR_EXPERIMENTO and not SOMENTE_RECOMBINAR

print('AMBIENTE: Google Colab',flush=True)
print('ENTRADAS: rede =', CAMINHO_JSON,'| personas =',CAMINHO_PERSONAS,flush=True)
print('REFERÊNCIA: NÃO É CONSULTADA NESTE NOTEBOOK | nome somente para auditoria:',NOME_REDE_BNREP,flush=True)
print('MODELO:',HF_MODEL_ID,'| revisão:',HF_MODEL_REVISION,flush=True)
print('PROTOCOLO:',METODO_HIBRIDO,'| confiança >=',LIMIAR_CONFIANCA,'=> FullDist; abaixo => Ranked Nodes',flush=True)
print('RESULTADOS PERSISTENTES:',DIR_ESTUDO,flush=True)
print('HF_HOME TEMPORÁRIO:',os.getenv('HF_HOME','padrão'),flush=True)

assert HF_MODEL_ID == 'Qwen/Qwen3.5-9B'
assert QWEN35_SEM_THINKING is True

## 1. Carregador Qwen 3.5 9B — uma única carga em GPU

Usa o carregamento multimodal oficial, mas somente entradas textuais e `enable_thinking=False`. As respostas textuais são justificativas geradas, não acesso ao raciocínio interno oculto.

In [ ]:
import numpy as np
import pandas as pd
from IPython.display import display
import networkx as nx
import matplotlib.pyplot as plt

from jinja2.exceptions import TemplateError
from pgmpy.factors.discrete import TabularCPD
try:
    from pgmpy.models import DiscreteBayesianNetwork as BayesianNetworkModel
except ImportError:
    from pgmpy.models import BayesianNetwork as BayesianNetworkModel

_TOKENIZER = None
_PROCESSOR = None
_MODEL = None
_GENERATOR = None
_CHAT_TEMPLATE_MODE = 'qwen35_multimodal_text_only'
_SYSTEM_FALLBACK_WARNED = False
_ULTIMA_GERACAO_META = {}


def carregar_llama():
    """Nome histórico da interface LLMAPI. Carrega SOMENTE Qwen3.5-9B (uma vez)."""
    global _TOKENIZER, _PROCESSOR, _MODEL, _GENERATOR
    if _MODEL is not None:
        return _MODEL
    import torch
    from transformers import AutoTokenizer, AutoProcessor, AutoModelForMultimodalLM, BitsAndBytesConfig
    from transformers import logging as hf_logging
    hf_logging.set_verbosity_warning()
    if not torch.cuda.is_available():
        raise RuntimeError('GPU CUDA necessária. No Colab, selecione um ambiente de execução com GPU.')
    print('[QWEN3.5-9B] Carregando:', HF_MODEL_ID,flush=True)
    dev=torch.cuda.get_device_properties(0)
    print('[GPU]',dev.name,'| VRAM: %.1f GiB' % (dev.total_memory/2**30),flush=True)
    major,_=torch.cuda.get_device_capability(0)
    dtype=torch.bfloat16 if major>=8 else torch.float16
    if dev.total_memory < 24*(2**30) and not USAR_4BITS:
        print('[AVISO] Menos de 24 GiB e BF16/FP16 sem quantização: risco elevado de OOM. '
              'Considere A100 40GB ou NF4 explícito.',flush=True)
    quant=(BitsAndBytesConfig(load_in_4bit=True,bnb_4bit_quant_type='nf4',
           bnb_4bit_use_double_quant=True,bnb_4bit_compute_dtype=dtype)
           if USAR_4BITS else None)
    token=os.getenv('HF_TOKEN') or None
    torch.manual_seed(SEED)
    inicio=time.perf_counter()
    print('[QWEN3.5-9B] Obtendo tokenizer e processor (cache ou download)...',flush=True)
    _TOKENIZER=AutoTokenizer.from_pretrained(HF_MODEL_ID,revision=HF_MODEL_REVISION,token=token)
    _PROCESSOR=AutoProcessor.from_pretrained(HF_MODEL_ID,revision=HF_MODEL_REVISION,token=token)
    print('[QWEN3.5-9B] Obtendo pesos; isso pode levar alguns minutos...',flush=True)
    try:
        opts=dict(revision=HF_MODEL_REVISION,token=token,device_map='auto',
                  dtype=dtype,low_cpu_mem_usage=True)
        if quant is not None:opts['quantization_config']=quant
        _MODEL=AutoModelForMultimodalLM.from_pretrained(HF_MODEL_ID,**opts)
        _MODEL.eval()
    except Exception as exc:
        print('[FALHA QWEN] Verifique versão recente de Transformers, CUDA, VRAM '
              'e compatibilidade da quantização NF4:',repr(exc),flush=True)
        raise
    _GENERATOR=_MODEL
    print('[QWEN3.5-9B] Modelo pronto em %.1fs | dtype=%s | NF4=%s' %
          (time.perf_counter()-inicio,dtype,USAR_4BITS),flush=True)
    return _MODEL


def _sample(system_prompt: str, user_prompt: str,
            temperature: float=TEMPERATURE, max_new_tokens: int=MAX_NEW_TOKENS_FULLDIST) -> str:
    """Geração text-only no modelo multimodal; preserva texto completo para auditoria."""
    global _CHAT_TEMPLATE_MODE, _SYSTEM_FALLBACK_WARNED, _ULTIMA_GERACAO_META
    import torch
    model=carregar_llama()
    messages=[]
    if system_prompt:
        messages.append({'role':'system','content':[{'type':'text','text':system_prompt}]})
    messages.append({'role':'user','content':[{'type':'text','text':user_prompt}]})
    args=dict(tokenize=True,add_generation_prompt=True,return_dict=True,
              return_tensors='pt',enable_thinking=not QWEN35_SEM_THINKING)
    try:
        inputs=_PROCESSOR.apply_chat_template(messages,**args)
        _CHAT_TEMPLATE_MODE='processor_text_only_nonthinking'
    except (ValueError,TypeError,TemplateError) as exc:
        # Algumas versões de template não aceitam papel system como bloco separado.
        # Preserva a persona, mas não omite a opção não-thinking.
        if not system_prompt:
            raise RuntimeError('Chat template Qwen3.5 incompatível; atualize Transformers.') from exc
        if not _SYSTEM_FALLBACK_WARNED:
            print('[AVISO] Template não aceitou papel system; movendo suas instruções para user:',
                  repr(exc),flush=True)
            _SYSTEM_FALLBACK_WARNED=True
        messages=[{'role':'user','content':[{'type':'text','text':
             f'INSTRUCTIONS:\n{system_prompt}\n\nTASK:\n{user_prompt}'}]}]
        inputs=_PROCESSOR.apply_chat_template(messages,**args)
        _CHAT_TEMPLATE_MODE='processor_system_text_in_user_nonthinking'
    if 'input_ids' not in inputs:
        raise RuntimeError('Processor Qwen3.5 não retornou input_ids para o prompt textual.')
    device=model.device
    inputs={k:(v.to(device) if hasattr(v,'to') else v) for k,v in inputs.items()}
    prompt_len=int(inputs['input_ids'].shape[-1])
    eos=getattr(model.generation_config,'eos_token_id',None) or _TOKENIZER.eos_token_id
    kw=dict(max_new_tokens=int(max_new_tokens),do_sample=bool(temperature>0),
            pad_token_id=_TOKENIZER.pad_token_id or _TOKENIZER.eos_token_id,
            eos_token_id=eos)
    if temperature>0:
        kw.update(temperature=temperature,top_p=1.0)
    inicio=time.perf_counter()
    with torch.inference_mode():
        ids=model.generate(**inputs,**kw)
    novos=ids[0][prompt_len:]
    full=_PROCESSOR.decode(novos,skip_special_tokens=True,
                            clean_up_tokenization_spaces=False).strip()
    _ULTIMA_GERACAO_META={'prompt_tokens':prompt_len,'tokens_gerados':int(novos.shape[-1]),
        'tempo_inferencia_segundos':round(time.perf_counter()-inicio,3),
        'thinking_habilitado':not QWEN35_SEM_THINKING}
    return full


class LLMAPI:
    call_counter = 0
    log_path = None  # inicializado após leitura de personas e criação de DIR_EXECUCAO

    @staticmethod
    def call(system_prompt: str, user_prompt: str,
             temperature: float=TEMPERATURE, max_retries: int=MAX_RETRIES,
             max_new_tokens: int=MAX_NEW_TOKENS_FULLDIST, context=None) -> str:
        if not REALIZAR_INFERENCIA:
            raise RuntimeError('A execução está bloqueada. Configure REALIZAR_INFERENCIA=True na seção 0.')
        LLMAPI.call_counter += 1
        call_id = LLMAPI.call_counter
        print(f'    [CONSULTA {call_id:05d}] Enviando ao modelo...', flush=True)
        last_exc = None
        context = context or {}
        for attempt in range(1, max_retries + 1):
            inicio = time.perf_counter()
            try:
                # Mesma semente para a mesma pergunta/rep em condições distintas.
                # Checkpoints retomados não alteram a sequência aleatória das próximas perguntas.
                import torch
                material=json.dumps({'prompt':user_prompt,'rep':context.get('rep',0),
                     'parse_tentativa':context.get('parse_tentativa',1),
                     'row_retry':context.get('row_retry',0),
                     'chamada_tentativa':attempt},
                     sort_keys=True,ensure_ascii=False)
                seed_call=(SEED+int(hashlib.sha256(material.encode()).hexdigest()[:8],16))%(2**32)
                torch.manual_seed(seed_call)
                if torch.cuda.is_available():torch.cuda.manual_seed_all(seed_call)
                response = _sample(system_prompt, user_prompt,
                                   temperature=temperature, max_new_tokens=max_new_tokens)
                audit = {'utc':datetime.now(timezone.utc).isoformat(), 'call_id':call_id,
                         'attempt':attempt, 'model':HF_MODEL_ID,
                         'session_id':ID_SESSAO,
                         'temperature':temperature, 'seed_consulta':seed_call,
                         'revisao_solicitada':HF_MODEL_REVISION,
                         'qwen35_sem_thinking':QWEN35_SEM_THINKING,
                         'estrategia_chat_template':_CHAT_TEMPLATE_MODE,
                         'context':context, 'system_prompt':system_prompt, 'response':response,
                         'prompt':user_prompt, 'seconds':time.perf_counter()-inicio,
                         'geracao_meta':dict(_ULTIMA_GERACAO_META)}
                with LLMAPI.log_path.open('a', encoding='utf-8') as f:
                    f.write(json.dumps(audit, ensure_ascii=False)+'\n')
                trecho = ' '.join(response.split())
                print(f'    [CONSULTA {call_id:05d}] OK, {audit["seconds"]:.1f}s | '
                      f'resposta: {trecho[-220:]}', flush=True)
                return response
            except (KeyboardInterrupt, SystemExit):
                raise
            except Exception as exc:
                last_exc = exc
                print(f'[WARN] Chamada {call_id} tentativa {attempt}/{max_retries}: {exc}')
                if attempt < max_retries:
                    time.sleep(min(2**attempt, 15))
        raise RuntimeError(f'Falha na chamada ao modelo {call_id}: {last_exc}')

## 2. Ler a rede fornecida (sem consultar a referência)

In [ ]:
# ==========================================================
# ENTRADA DOS NÓS
# ==========================================================
def load_bn_spec(json_path: str) -> Tuple[Dict[str, Dict[str, Any]], List[Tuple[str, str]]]:
    """
    Lê um JSON contendo:
    - network: metadados gerais da rede, opcional
    - nodes: nós em formato antigo ou novo
    - edges: relações causais já definidas pelo usuário

    Formato antigo aceito:
    {
      "nodes": {
        "A": {
          "description": "Descrição do nó A.",
          "states": ["False", "True"]
        }
      },
      "edges": [...]
    }

    Formato novo aceito:
    {
      "network": {...},
      "nodes": [
        {
          "name": "A",
          "definition": "...",
          "entity": "...",
          "scale_type": "ordinal",
          "states": [
            {"name": "Low", "meaning": "..."},
            {"name": "High", "meaning": "..."}
          ],
          "temporal_context": "...",
          "measurement_context": "...",
          "notes": "..."
        }
      ],
      "edges": [...]
    }
    """
    path = Path(json_path)

    # utf-8-sig evita erro quando o arquivo vier com BOM
    with path.open("r", encoding="utf-8-sig") as f:
        data = json.load(f)

    raw_nodes = data.get("nodes")
    raw_edges = data.get("edges", [])
    network_info = data.get("network", {})

    if raw_nodes is None:
        raise ValueError("O JSON precisa conter o campo 'nodes'.")

    nodes: Dict[str, Dict[str, Any]] = {}

    # ======================================================
    # CASO 1: formato antigo
    # "nodes": { "A": {"description": "...", "states": [...] } }
    # ======================================================
    if isinstance(raw_nodes, dict):
        for name, value in raw_nodes.items():
            name = str(name).strip()

            if isinstance(value, str):
                nodes[name] = {
                    "description": value.strip(),
                    "states": ["False", "True"],
                }

            elif isinstance(value, dict):
                description = str(value.get("description", name)).strip()
                states = value.get("states", ["False", "True"])

                if not isinstance(states, list) or len(states) < 2:
                    raise ValueError(f"O nó '{name}' precisa ter pelo menos dois estados.")

                nodes[name] = {
                    "description": description,
                    "states": [str(s).strip() for s in states],
                }

            else:
                raise ValueError(f"Formato inválido para o nó '{name}'.")

    # ======================================================
    # CASO 2: formato novo
    # "nodes": [ {"name": "...", "definition": "...", ... } ]
    # ======================================================
    elif isinstance(raw_nodes, list):
        for node_obj in raw_nodes:
            if not isinstance(node_obj, dict):
                raise ValueError(f"Cada item de 'nodes' precisa ser um objeto. Recebido: {node_obj}")

            name = str(node_obj.get("name", "")).strip()
            if not name:
                raise ValueError(f"Nó sem campo 'name': {node_obj}")

            raw_states = node_obj.get("states", [])
            if not isinstance(raw_states, list) or len(raw_states) < 2:
                raise ValueError(f"O nó '{name}' precisa ter pelo menos dois estados.")

            state_names = []
            state_descriptions = []

            for state in raw_states:
                if isinstance(state, dict):
                    state_name = str(state.get("name", "")).strip()
                    state_meaning = str(state.get("meaning", "")).strip()

                    if not state_name:
                        raise ValueError(f"Estado sem campo 'name' no nó '{name}': {state}")

                    state_names.append(state_name)

                    if state_meaning:
                        state_descriptions.append(f"{state_name}: {state_meaning}")

                else:
                    state_name = str(state).strip()
                    state_names.append(state_name)

            description_parts = []

            network_domain = str(network_info.get("domain", "")).strip()
            network_description = str(network_info.get("description", "")).strip()

            if network_domain:
                description_parts.append(f"Network domain: {network_domain}.")

            if network_description:
                description_parts.append(f"Network description: {network_description}")

            definition = str(node_obj.get("definition", "")).strip()
            entity = str(node_obj.get("entity", "")).strip()
            scale_type = str(node_obj.get("scale_type", "")).strip()
            temporal_context = str(node_obj.get("temporal_context", "")).strip()
            measurement_context = str(node_obj.get("measurement_context", "")).strip()
            notes = str(node_obj.get("notes", "")).strip()

            if definition:
                description_parts.append(f"Definition: {definition}")

            if entity:
                description_parts.append(f"Entity: {entity}")

            if scale_type:
                description_parts.append(f"Scale type: {scale_type}")

            if state_descriptions:
                description_parts.append("State meanings: " + "; ".join(state_descriptions))

            if temporal_context:
                description_parts.append(f"Temporal context: {temporal_context}")

            if measurement_context:
                description_parts.append(f"Measurement context: {measurement_context}")

            if notes:
                description_parts.append(f"Notes: {notes}")

            description = " ".join(description_parts).strip()

            if not description:
                description = name

            nodes[name] = {
                "description": description,
                "states": state_names,
                "raw_metadata": node_obj,
            }

    else:
        raise ValueError("'nodes' deve ser um objeto ou uma lista.")

    edges = parse_and_validate_edges(raw_edges, nodes)

    return nodes, edges

def parse_and_validate_edges(
    raw_edges: List[Any],
    nodes_info: Dict[str, Dict[str, Any]],
) -> List[Tuple[str, str]]:
    """
    Converte e valida as arestas fornecidas no JSON.

    Aceita:
    - {"source": "A", "target": "B"}
    - ["A", "B"]

    Retorna:
    - [("A", "B"), ...]
    """
    known_nodes = set(nodes_info.keys())
    edges: List[Tuple[str, str]] = []

    if not isinstance(raw_edges, list):
        raise ValueError("'edges' deve ser uma lista.")

    for e in raw_edges:
        if isinstance(e, dict):
            src = str(e.get("source", "")).strip()
            dst = str(e.get("target", "")).strip()

        elif isinstance(e, (list, tuple)) and len(e) >= 2:
            src = str(e[0]).strip()
            dst = str(e[1]).strip()

        else:
            raise ValueError(f"Aresta inválida: {e}")

        if src not in known_nodes:
            raise ValueError(f"Aresta inválida: nó de origem desconhecido: {src}")

        if dst not in known_nodes:
            raise ValueError(f"Aresta inválida: nó de destino desconhecido: {dst}")

        if src == dst:
            raise ValueError(f"Aresta inválida: self-loop detectado: {src} -> {dst}")

        edges.append((src, dst))

    # Remove duplicatas mantendo a ordem
    deduped = []
    seen = set()

    for edge in edges:
        if edge not in seen:
            seen.add(edge)
            deduped.append(edge)

    # Verifica se é DAG
    graph = nx.DiGraph()
    graph.add_nodes_from(nodes_info.keys())
    graph.add_edges_from(deduped)

    if not nx.is_directed_acyclic_graph(graph):
        raise ValueError("As arestas fornecidas geram ciclo. A Rede Bayesiana precisa ser um DAG.")

    return deduped


def node_context_text(
    nodes_info: Dict[str, Dict[str, Any]],
    only_nodes: Optional[List[str]] = None
) -> str:
    selected = only_nodes if only_nodes is not None else list(nodes_info.keys())

    lines = []

    for node in selected:
        info = nodes_info[node]

        states = ", ".join(info["states"])

        lines.append(
            f"- {node}: {info['description']} Possible states: [{states}]"
        )

    return "\n".join(lines)

## 3. Ler as três personas definidas pelo pesquisador

O arquivo `personas.json` continua sendo usado. Nesta versão, o notebook exige **exatamente três personas ativas** e executa quatro condições independentes: `controle`, `persona 1`, `persona 2` e `persona 3`. Os pesos existentes no JSON são registrados, mas não alteram a geração individual de cada rede.


In [ ]:
# ============================================================
# 3.1 — Personas: exatamente 3 ativas + controle
# ============================================================
PERSONAS_CONFIG = None
PERSONAS_ATIVAS = []
PERSONAS_HASH = None

def carregar_personas_json(path, nodes_info):
    path=Path(path)
    if not path.is_file():
        raise FileNotFoundError(f'JSON de personas não encontrado: {path}.')
    dados=json.loads(path.read_text(encoding='utf-8-sig'))
    if not isinstance(dados,dict) or not isinstance(dados.get('personas'),list):
        raise ValueError('personas.json precisa conter a lista "personas".')

    ids=set()
    personas=[]
    for p in dados['personas']:
        if not isinstance(p,dict):
            raise ValueError('Toda persona precisa ser um objeto JSON.')
        pid=p.get('id','')
        if not isinstance(pid,str) or not re.fullmatch(r'[A-Za-z0-9_-]{1,60}',pid):
            raise ValueError(f'ID de persona inválido: {pid!r}.')
        if pid.lower() in ('controle','ensemble','uniform','random') or pid in ids:
            raise ValueError('ID de persona repetido/reservado: '+pid)
        ids.add(pid)
        if not isinstance(p.get('ativo',True),bool):
            raise ValueError('"ativo" deve ser booleano em '+pid)
        if not p.get('ativo',True):
            continue

        papel=p.get('papel','')
        especialidade=p.get('especialidade','')
        instrucoes=p.get('instrucoes','')
        if not all(isinstance(x,str) for x in (papel,especialidade,instrucoes)):
            raise ValueError('papel, especialidade e instrucoes devem ser strings em '+pid)
        if not papel.strip() or not instrucoes.strip():
            raise ValueError('Persona ativa precisa de papel e instrucoes: '+pid)

        w=p.get('peso',0.0)
        if isinstance(w,bool) or not isinstance(w,(int,float)) or not np.isfinite(w) or w<0:
            raise ValueError('peso inválido em '+pid)

        personas.append({
            'id':pid,
            'ativo':True,
            'papel':papel.strip(),
            'especialidade':especialidade.strip(),
            'instrucoes':instrucoes.strip(),
            'peso':float(w),
        })

    if EXIGIR_TRES_PERSONAS and len(personas)!=3:
        raise ValueError(f'Este protocolo exige exatamente 3 personas ativas; encontrado: {len(personas)}.')
    if not personas:
        raise ValueError('Ao menos uma persona ativa é necessária.')

    controle=dados.get('controle',{'ativo':True})
    if not isinstance(controle,dict) or not isinstance(controle.get('ativo',True),bool):
        raise ValueError('controle precisa de campo "ativo" booleano.')
    if not controle.get('ativo',True):
        raise ValueError('Este protocolo exige controle.ativo=true.')

    calib=dados.get('calibracao',{})
    if not isinstance(calib,dict):
        raise ValueError('calibracao deve ser um objeto.')
    repeticoes=calib.get('repeticoes',1)
    if type(repeticoes) is not int or not 1<=repeticoes<=20:
        raise ValueError('calibracao.repeticoes deve ser inteiro de 1 a 20.')
    somente=calib.get('somente_recombinar',False)
    if not isinstance(somente,bool):
        raise ValueError('calibracao.somente_recombinar deve ser booleano.')

    metodos_antigos=calib.get('metodos',[])
    pesos_por_no=calib.get('pesos_por_no',{})
    if pesos_por_no and not isinstance(pesos_por_no,dict):
        raise ValueError('pesos_por_no deve ser objeto.')

    print('[PERSONAS] 3 personas ativas + controle:',flush=True)
    for p in personas:
        print(f"  {p['id']:22s} peso_documental={p['peso']:.4f} | papel={p['papel']}",flush=True)
    if metodos_antigos:
        print('[INFO] calibracao.metodos do JSON é ignorado nesta versão. Método fixo:',METODO_HIBRIDO,flush=True)
    print('[INFO] Pesos de persona são somente metadados; nenhuma fusão é feita na geração.',flush=True)

    return {
        'personas':personas,
        'controle':True,
        'repeticoes':repeticoes,
        'pesos_por_no':pesos_por_no,
        'somente_recombinar':somente,
        'hash':hashlib.sha256(path.read_bytes()).hexdigest(),
        'arquivo':str(path.resolve()),
    }


def prompt_sistema_persona(p):
    if p is None:
        return ''
    partes=[f"You are acting in the role of: {p['papel']}."]
    if p.get('especialidade'):
        partes.append(f"Specialty: {p['especialidade']}.")
    partes.append('Additional persona instructions supplied by the researcher: '+p['instrucoes'])
    partes.append('Follow the exact output format requested by the user prompt. Report uncertainty honestly.')
    return '\n'.join(partes)


## 4. Prompts: triagem de confiança + FullDist

Para cada nó com pais, o Qwen recebe **uma única chamada de triagem** contendo todas as combinações dos pais. Nessa chamada ele não fornece probabilidades da CPT: apenas declara quanta confiança possui para estimar cada linha e, adicionalmente, a direção qualitativa esperada de cada pai (`positive`, `negative` ou `uncertain`).

Somente as linhas com `confidence >= LIMIAR_CONFIANCA` recebem uma chamada FullDist. Essas distribuições são as âncoras usadas no ajuste do Ranked Nodes.


In [ ]:
# ----------------------------------------------------------
# PROMPTS E PARSERS — confiança + FullDist
# ----------------------------------------------------------
def _descricoes_relevantes(nodes_info, node, parents):
    return node_context_text(nodes_info, [node] + list(parents))


def _condicao(parents, parent_state_values):
    return (', '.join(f'{p} is {v}' for p, v in zip(parents, parent_state_values))
            if parents else 'there are no parent conditions')


def format_epk_prompt_full_distribution(nodes_info, node, parents, parent_state_values):
    # Mantém o template FullDist do notebook anterior para comparabilidade das âncoras.
    desc = _descricoes_relevantes(nodes_info, node, parents)
    order = ', '.join(nodes_info[node]['states'])
    pergunta = (f'Given that {_condicao(parents, parent_state_values)}, '
                f'what is the probability distribution of {node}?')
    return f"""These nodes are related to the question inside a Bayesian Network:

{desc}

Given this information, answer the following question by providing the probability distribution of the node. Your answer should include your reasoning and at the end a sentence that says 'The probability distribution of the node is:' followed by the probabilities given in a tuple with each probability representing a state in the given order.

Order of states: ({order})

{pergunta}"""


def format_prompt_confianca_no(nodes_info, node, parents, parent_combinations):
    desc=_descricoes_relevantes(nodes_info,node,parents)
    linhas='\n'.join(
        f'{i}: '+', '.join(f'{p}={v}' for p,v in zip(parents,comb))
        for i,comb in enumerate(parent_combinations)
    )
    pais=', '.join(parents)
    return f"""These variables belong to a Bayesian Network:

{desc}

Before estimating any numerical CPT probabilities, assess whether your INTERNAL KNOWLEDGE is sufficient to make an informed conditional probability estimate for each parent configuration of node {node}.

Important definition of confidence:
- confidence measures how much relevant knowledge you believe you have for estimating the FULL probability distribution;
- it does NOT mean how easy it is to invent a plausible numeric answer;
- use values from 0 to 1;
- do not provide any CPT probabilities in this response.

Also report the qualitative influence of each parent on higher states of {node}: "positive", "negative", or "uncertain". This direction is only a qualitative aid for Ranked Nodes.

Parents: {pais}
Configurations:
{linhas}

Return ONLY one valid JSON object in exactly this structure:
{{
  "parent_influence": [
    {{"parent": "PARENT_NAME", "direction": "positive|negative|uncertain", "confidence": 0.0}}
  ],
  "rows": [
    {{"row": 0, "confidence": 0.0}}
  ]
}}

Include every parent exactly once and every row index from 0 to {len(parent_combinations)-1} exactly once. Do not include explanations outside the JSON."""


_NUMERO_PROB = (r'(?P<num>(?:\d+(?:\.\d*)?|\.\d+)'
                r'(?:\s*/\s*(?:\d+(?:\.\d*)?|\.\d+))?)'
                r'(?P<percent>\s*%)?(?![\w/])')


def _interpretar_numero_prob(match):
    valor = match.group('num').strip()
    if '/' in valor:
        a,b=valor.split('/',1)
        den=float(b.strip())
        if den==0:
            raise ValueError('Fração com denominador zero.')
        prob=float(a.strip())/den
    else:
        prob=float(valor)
    if match.group('percent'):
        prob/=100
    if not np.isfinite(prob) or not 0<=prob<=1:
        raise ValueError('Probabilidade fora de [0,1]: '+str(prob))
    return float(prob)


def extrair_distribuicao(texto, n_estados, estados=None):
    def validar_tupla(conteudo):
        partes=[p.strip() for p in conteudo.split(',')]
        if len(partes)!=n_estados:
            return None
        saida=[]
        for p in partes:
            num=re.fullmatch(_NUMERO_PROB,p)
            if num is None:
                return None
            saida.append(_interpretar_numero_prob(num))
        return saida if sum(saida)>0 else None

    marcadores=[
        r'the\s+probability\s+distribution\s+of\s+the\s+node\s+is\s*[:=]',
        r'the\s+(?:final|estimated)\s+probability\s+distribution\s+is\s*[:=]',
        r'(?:final\s+)?probabilities\s+in\s+order\s*[:=]',
        r'final\s+distribution\s*[:=]',
    ]
    marcados=[mt for padrao in marcadores for mt in re.finditer(padrao,texto,re.I)]
    if marcados:
        conclusao=texto[max(marcados,key=lambda mt:mt.start()).end():]
        tupla=re.search(r'[\(\[]\s*([^\)\]]+?)\s*[\)\]]',conclusao)
        if tupla and tupla.start()<240:
            valores=validar_tupla(tupla.group(1))
            if valores is not None:
                return valores

    linhas=[linha.strip() for linha in texto.splitlines() if linha.strip()]
    if linhas:
        ultima=re.sub(r'^\s*(?:final answer|answer)\s*[:=]\s*','',linhas[-1],flags=re.I)
        mt=re.fullmatch(r'[\(\[]\s*([^\)\]]+)\s*[\)\]]\s*[.\s]*',ultima)
        if mt:
            valores=validar_tupla(mt.group(1))
            if valores is not None:
                return valores

    if estados and len(estados)==n_estados:
        bloco=texto[-600:].strip().rstrip('.')
        num=r'(?:\d+(?:\.\d*)?|\.\d+)(?:\s*/\s*(?:\d+(?:\.\d*)?|\.\d+))?\s*%?'
        padrao=r'^\s*'+r'\s*[,;\n]\s*'.join(
            rf'{re.escape(s)}\s*[:=]\s*({num})' for s in estados)+r'\s*$'
        linhas=bloco.splitlines()
        alternativas=[bloco,'\n'.join(linhas[-n_estados:]),linhas[-1]]
        for candidato in alternativas:
            candidato=re.sub(r'^\s*(?:final\s+distribution|probabilities\s+in\s+order)\s*[:=]\s*',
                              '',candidato,flags=re.I)
            mt=re.fullmatch(padrao,candidato,flags=re.I)
            if mt:
                nums=[]
                for campo in mt.groups():
                    v=re.fullmatch(_NUMERO_PROB,campo.strip())
                    if v is None:
                        break
                    nums.append(_interpretar_numero_prob(v))
                if len(nums)==n_estados and sum(nums)>0:
                    return nums

    raise ValueError('FullDist: distribuição final ambígua; não extrair valores intermediários.')


def normalize_distribution(raw_probs):
    arr=np.asarray(raw_probs,dtype=float)
    if arr.ndim!=1 or arr.size<2 or np.any(~np.isfinite(arr)) or np.any(arr<0):
        raise ValueError('Distribuição inválida.')
    s=arr.sum()
    if s<=0:
        raise ValueError('Probabilidades todas zero.')
    return (arr/s).tolist()


def _extrair_objeto_json(texto):
    texto=texto.strip()
    texto=re.sub(r'^```(?:json)?\s*','',texto,flags=re.I)
    texto=re.sub(r'\s*```$','',texto)
    try:
        return json.loads(texto)
    except Exception:
        inicio=texto.find('{')
        fim=texto.rfind('}')
        if inicio>=0 and fim>inicio:
            return json.loads(texto[inicio:fim+1])
        raise ValueError('Nenhum objeto JSON válido encontrado.')


def extrair_confiancas(texto, parents, n_rows):
    obj=_extrair_objeto_json(texto)
    if not isinstance(obj,dict):
        raise ValueError('Resposta de confiança precisa ser objeto JSON.')
    rows=obj.get('rows')
    influences=obj.get('parent_influence')
    if not isinstance(rows,list) or not isinstance(influences,list):
        raise ValueError('JSON precisa conter rows e parent_influence.')

    mapa={}
    for x in rows:
        if not isinstance(x,dict) or type(x.get('row')) is not int:
            raise ValueError('Cada row precisa de índice inteiro.')
        idx=x['row']
        c=x.get('confidence')
        if idx in mapa or idx<0 or idx>=n_rows:
            raise ValueError('Índice de row inválido/repetido: '+str(idx))
        if isinstance(c,bool) or not isinstance(c,(int,float)) or not np.isfinite(c) or not 0<=c<=1:
            raise ValueError('Confiança inválida na row '+str(idx))
        mapa[idx]=float(c)
    if set(mapa)!=set(range(n_rows)):
        raise ValueError('A resposta não contém exatamente todas as linhas esperadas.')

    infl={}
    for x in influences:
        if not isinstance(x,dict):
            raise ValueError('parent_influence inválido.')
        p=x.get('parent')
        d=str(x.get('direction','')).lower().strip()
        c=x.get('confidence')
        if p not in parents or p in infl:
            raise ValueError('Pai ausente/desconhecido/repetido: '+str(p))
        if d not in ('positive','negative','uncertain'):
            raise ValueError('Direção inválida em '+str(p))
        if isinstance(c,bool) or not isinstance(c,(int,float)) or not np.isfinite(c) or not 0<=c<=1:
            raise ValueError('Confiança de direção inválida em '+str(p))
        infl[p]={'direction':d,'confidence':float(c)}
    if set(infl)!=set(parents):
        raise ValueError('parent_influence precisa conter todos os pais exatamente uma vez.')

    return mapa,infl


def registrar_evento_parser(metodo, contexto, tentativa, resposta, aceito,
                             valor=None, regra=None, erro=None):
    contexto=dict(contexto or {})
    evento={
        'utc':datetime.now(timezone.utc).isoformat(),
        'session_id':ID_SESSAO,
        'metodo':metodo,
        'persona_id':contexto.get('persona_id','controle'),
        'repeticao':contexto.get('rep',0)+1,
        'no':contexto.get('node'),
        'pais':contexto.get('pais'),
        'tentativa_parser':tentativa,
        'aceita':bool(aceito),
        'valor':valor,
        'regra_extracao':regra,
        'erro':str(erro) if erro is not None else None,
        'resposta_integral':resposta,
    }
    caminho=globals().get('PARSER_LOG_PATH')
    if caminho is not None:
        Path(caminho).parent.mkdir(parents=True,exist_ok=True)
        with Path(caminho).open('a',encoding='utf-8') as f:
            f.write(json.dumps(evento,ensure_ascii=False,default=str)+'\n')
    return evento


def query_llm_full_distribution(prompt,n_estados,temperature=TEMPERATURE,
                                verbose=False,max_retries=MAX_RETRIES,system_prompt='',context=None):
    context=dict(context or {})
    for tentativa in range(1,max_retries+1):
        raw=LLMAPI.call(
            system_prompt=system_prompt,
            user_prompt=prompt,
            temperature=temperature,
            max_new_tokens=MAX_NEW_TOKENS_FULLDIST if tentativa==1 else MAX_NEW_TOKENS_FULLDIST_RETRY,
            context={**context,'parse_tentativa':tentativa},
        )
        try:
            arr=extrair_distribuicao(raw,n_estados,estados=context.get('states'))
            arr=normalize_distribution(arr)
            registrar_evento_parser('FullDist',context,tentativa,raw,True,arr,'distribuicao_final')
            print(f'    [EXTRAÇÃO FullDist] {context.get("node","?")}: {arr}',flush=True)
            if verbose:
                print('PROMPT:',prompt,'\nRESPOSTA:',raw,'\nvalores:',arr)
            return arr,raw
        except ValueError as exc:
            registrar_evento_parser('FullDist',context,tentativa,raw,False,erro=exc)
            print(f'[WARN] FullDist tentativa {tentativa}/{max_retries}: {exc}',flush=True)
            if tentativa==max_retries:
                raise


def query_llm_confianca(prompt,parents,n_rows,system_prompt='',context=None):
    context=dict(context or {})
    for tentativa in range(1,MAX_RETRIES_CONFIANCA+1):
        raw=LLMAPI.call(
            system_prompt=system_prompt,
            user_prompt=prompt,
            temperature=0.0,
            max_new_tokens=MAX_NEW_TOKENS_CONFIANCA,
            context={**context,'parse_tentativa':tentativa,'fase':'triagem_confianca'},
        )
        try:
            conf,infl=extrair_confiancas(raw,parents,n_rows)
            registrar_evento_parser(
                'ConfidenceScreen',
                context,
                tentativa,
                raw,
                True,
                {'rows':conf,'parent_influence':infl},
                'json_estruturado',
            )
            return conf,infl,raw
        except (ValueError,json.JSONDecodeError) as exc:
            registrar_evento_parser('ConfidenceScreen',context,tentativa,raw,False,erro=exc)
            print(f'[WARN] Confiança tentativa {tentativa}/{MAX_RETRIES_CONFIANCA}: {exc}',flush=True)
            if tentativa==MAX_RETRIES_CONFIANCA:
                raise


## 5. Construção híbrida das CPTs

O Ranked Nodes usado aqui é uma implementação **WMEAN**. Cada estado ordinal ocupa um intervalo igualmente espaçado em `[0,1]`. Os pais são combinados por uma média ponderada; a direção pode ser positiva ou negativa. Uma Normal truncada ao intervalo `[0,1]` transforma a média em probabilidades para os estados do nó filho.

Os pesos e o parâmetro de dispersão `sigma` são ajustados para aproximar as distribuições FullDist das linhas de alta confiança, usando **Hellinger quadrático ponderado pela confiança**. Quando há poucas âncoras, o ajuste usa regularização e as direções qualitativas declaradas na triagem. Nenhuma CPT de referência é usada nesse ajuste.


In [ ]:
# ----------------------------------------------------------
# RANKED NODES + GERAÇÃO HÍBRIDA
# ----------------------------------------------------------
from scipy.optimize import minimize
from scipy.special import ndtr


def _scale_type_ordinal(info):
    meta=info.get('raw_metadata',{}) if isinstance(info,dict) else {}
    st=str(meta.get('scale_type','')).strip().lower() if isinstance(meta,dict) else ''
    if st in ('ordinal','ordered','ranked'):
        return True
    return bool(ASSUMIR_ORDEM_ESTADOS_SEM_METADADO)


def _rn_elegivel(nodes_info,node,parents):
    if not parents:
        return False
    return _scale_type_ordinal(nodes_info[node]) and all(_scale_type_ordinal(nodes_info[p]) for p in parents)


def _midpoint_estado(nodes_info,node,state):
    estados=nodes_info[node]['states']
    i=estados.index(state)
    k=len(estados)
    return (i+0.5)/k


def _rn_probs(mu,sigma,k):
    sigma=float(np.clip(sigma,RN_SIGMA_MIN,RN_SIGMA_MAX))
    mu=float(np.clip(mu,0.0,1.0))
    bounds=np.linspace(0.0,1.0,k+1)
    den=ndtr((1.0-mu)/sigma)-ndtr((0.0-mu)/sigma)
    if not np.isfinite(den) or den<=RN_EPS:
        arr=np.ones(k)/k
        return arr

    vals=[]
    for a,b in zip(bounds[:-1],bounds[1:]):
        vals.append((ndtr((b-mu)/sigma)-ndtr((a-mu)/sigma))/den)

    arr=np.asarray(vals,dtype=float)
    arr=np.clip(arr,RN_EPS,None)
    arr/=arr.sum()
    return arr


def _softmax(z):
    z=np.asarray(z,dtype=float)
    z=z-z.max()
    e=np.exp(z)
    return e/e.sum()


def _hellinger2(p,q):
    p=np.asarray(p,dtype=float)
    q=np.asarray(q,dtype=float)
    return float(0.5*np.sum((np.sqrt(np.clip(p,0,None))-np.sqrt(np.clip(q,0,None)))**2))


def _direction_from_hint(influencia,parent):
    d=influencia.get(parent,{}).get('direction','uncertain')
    return -1 if d=='negative' else 1


def _mu_linha(nodes_info,parents,comb,weights,directions):
    xs=[]
    for p,st,d in zip(parents,comb,directions):
        x=_midpoint_estado(nodes_info,p,st)
        xs.append(x if d>0 else 1.0-x)
    return float(np.dot(weights,np.asarray(xs,dtype=float)))


def ajustar_ranked_nodes(nodes_info,node,parents,pcombs,ancoras,influencia):
    p=len(parents)
    k=len(nodes_info[node]['states'])
    if p==0:
        raise ValueError('Ranked Nodes exige ao menos um pai.')

    hints=tuple(_direction_from_hint(influencia,pai) for pai in parents)

    # Sem âncoras: usa apenas informação qualitativa da triagem + pesos iguais.
    if not ancoras:
        return {
            'weights':[1.0/p]*p,
            'directions':list(hints),
            'sigma':RN_SIGMA_DEFAULT,
            'loss':None,
            'fit_status':'sem_ancoras_pesos_iguais_direcao_triagem',
            'n_anchors':0,
        }

    # Com poucas âncoras, mantém as direções declaradas.
    # Com amostra suficiente, testa direções possíveis e deixa as âncoras decidirem.
    minimo_para_direcao=max(2,p+1)
    if len(ancoras)>=minimo_para_direcao and p<=RN_ENUMERAR_DIRECOES_ATE:
        candidatos=list(itertools.product((1,-1),repeat=p))
    else:
        candidatos=[hints]

    conf=np.asarray([max(float(a['confidence']),RN_EPS) for a in ancoras],dtype=float)
    conf=conf/conf.sum()
    melhores=None
    log_lo,log_hi=np.log(RN_SIGMA_MIN),np.log(RN_SIGMA_MAX)

    for directions in candidatos:
        def objective(theta):
            weights=_softmax(theta[:p])
            sigma=float(np.exp(theta[p]))
            erro=0.0
            for w_anchor,a in zip(conf,ancoras):
                mu=_mu_linha(nodes_info,parents,a['comb'],weights,directions)
                pred=_rn_probs(mu,sigma,k)
                erro+=float(w_anchor)*_hellinger2(a['distribution'],pred)
            reg=RN_REGULARIZACAO_PESOS*float(np.sum((weights-(1.0/p))**2))
            return erro+reg

        x0=np.r_[np.zeros(p),np.log(RN_SIGMA_DEFAULT)]
        bounds=[(-6,6)]*p+[(log_lo,log_hi)]
        res=minimize(objective,x0,method='L-BFGS-B',bounds=bounds,options={'maxiter':500})

        rec={
            'weights':_softmax(res.x[:p]).tolist(),
            'directions':list(directions),
            'sigma':float(np.exp(res.x[p])),
            'loss':float(res.fun),
            'fit_status':'ajustado' if res.success else 'ajustado_com_aviso_'+str(res.message),
            'n_anchors':len(ancoras),
        }
        if melhores is None or rec['loss']<melhores['loss']-1e-12:
            melhores=rec

    return melhores


def _fingerprint_hibrido(nodes_info,edges,persona,repeticao):
    b=json.dumps({
        'nodes':nodes_info,
        'edges':edges,
        'modelo':HF_MODEL_ID,
        'revision':HF_MODEL_REVISION,
        'temperature':TEMPERATURE,
        '4bits':USAR_4BITS,
        'seed':SEED,
        'metodo':METODO_HIBRIDO,
        'limiar_confianca':LIMIAR_CONFIANCA,
        'rn_sigma_default':RN_SIGMA_DEFAULT,
        'rn_sigma_min':RN_SIGMA_MIN,
        'rn_sigma_max':RN_SIGMA_MAX,
        'rn_reg':RN_REGULARIZACAO_PESOS,
        'persona':({k:persona.get(k) for k in ('id','papel','especialidade','instrucoes')} if persona else None),
        'repeticao':repeticao,
        'schema':'confidence_rn_v1',
    },sort_keys=True,ensure_ascii=False,default=str).encode()
    return hashlib.sha256(b).hexdigest()[:16]


def construir_rede_vazia(nodes_info,edges):
    model=BayesianNetworkModel(edges)
    model.add_nodes_from(nodes_info.keys())
    return model


def gerar_cpts_confidence_rn(model,nodes_info,persona=None,repeticao=0,checkpoint_dir=None,verbose=True):
    global PARSER_LOG_PATH

    etiqueta=persona['id'] if persona else 'controle'
    system_prompt=prompt_sistema_persona(persona)
    checkpoint_dir=Path(checkpoint_dir or DIR_ESTUDO/'checkpoints_confidence_rn')
    checkpoint_dir.mkdir(parents=True,exist_ok=True)

    fingerprint=_fingerprint_hibrido(nodes_info,list(model.edges()),persona,repeticao)
    audit_dir=DIR_ESTUDO/'auditoria_confidence_rn'
    audit_dir.mkdir(parents=True,exist_ok=True)
    LLMAPI.log_path=audit_dir/f'consultas_{fingerprint}.jsonl'
    PARSER_LOG_PATH=audit_dir/f'parser_{fingerprint}.jsonl'

    detalhes_dir=checkpoint_dir/fingerprint
    detalhes_dir.mkdir(parents=True,exist_ok=True)

    cpds=[]
    metadados={
        'condicao':etiqueta,
        'metodo':METODO_HIBRIDO,
        'limiar_confianca':LIMIAR_CONFIANCA,
        'nodes':{},
        'fingerprint':fingerprint,
    }
    inicio=time.perf_counter()

    for indice_no,node in enumerate(model.nodes(),start=1):
        parents=list(model.get_parents(node))
        states=nodes_info[node]['states']
        pcombs=list(itertools.product(*[nodes_info[p]['states'] for p in parents])) if parents else [()]
        ordinal=_rn_elegivel(nodes_info,node,parents)

        print(
            f'\n[NÓ {indice_no}/{len(model.nodes())}] {node} | '
            f'pais={parents or "nenhum"} | linhas={len(pcombs)} | RN={ordinal}',
            flush=True,
        )

        node_meta={
            'parents':parents,
            'states':states,
            'ordinal_rn_elegivel':ordinal,
            'rows':[],
        }

        # Nó raiz: uma única distribuição; RN não reduz custo aqui.
        if not parents:
            arquivo=detalhes_dir/f'{re.sub("[^A-Za-z0-9_-]","_",node)}_root.json'
            if arquivo.is_file():
                reg=json.loads(arquivo.read_text(encoding='utf-8'))
                dist=reg['distribution']
                print('  [CHECKPOINT ROOT]',dist,flush=True)
            else:
                prompt=format_epk_prompt_full_distribution(nodes_info,node,parents,())
                dist,raw=query_llm_full_distribution(
                    prompt,
                    len(states),
                    system_prompt=system_prompt,
                    context={
                        'persona_id':etiqueta,
                        'rep':repeticao,
                        'node':node,
                        'pais':{},
                        'states':states,
                        'fase':'root_fullDist',
                    },
                )
                reg={
                    'node':node,
                    'row':0,
                    'parents':{},
                    'confidence':None,
                    'source':'LLM_FULLDIST_ROOT',
                    'distribution':dist,
                    'raw':raw,
                    'prompt':prompt,
                    'utc':datetime.now(timezone.utc).isoformat(),
                }
                _salvar_json_atomico(arquivo,reg)

            linhas=[dist]
            node_meta['rows']=[{
                'row':0,
                'parents':{},
                'confidence':None,
                'source':'LLM_FULLDIST_ROOT',
                'distribution':dist,
            }]
            node_meta['rn_parameters']=None

        # Nó não ordinal: não usar Ranked Nodes silenciosamente.
        elif not ordinal:
            linhas=[]
            print('  [FALLBACK] Nó/pais não marcados como ordinal; FullDist em todas as linhas.',flush=True)

            for ci,comb in enumerate(pcombs):
                arquivo=detalhes_dir/f'{re.sub("[^A-Za-z0-9_-]","_",node)}_fallback_{ci}.json'
                if arquivo.is_file():
                    reg=json.loads(arquivo.read_text(encoding='utf-8'))
                    dist=reg['distribution']
                else:
                    prompt=format_epk_prompt_full_distribution(nodes_info,node,parents,comb)
                    dist,raw=query_llm_full_distribution(
                        prompt,
                        len(states),
                        system_prompt=system_prompt,
                        context={
                            'persona_id':etiqueta,
                            'rep':repeticao,
                            'node':node,
                            'pais':dict(zip(parents,comb)),
                            'states':states,
                            'fase':'nonordinal_fullDist',
                        },
                    )
                    reg={
                        'node':node,
                        'row':ci,
                        'parents':dict(zip(parents,comb)),
                        'confidence':None,
                        'source':'LLM_FULLDIST_FALLBACK_NONORDINAL',
                        'distribution':dist,
                        'raw':raw,
                        'prompt':prompt,
                        'utc':datetime.now(timezone.utc).isoformat(),
                    }
                    _salvar_json_atomico(arquivo,reg)

                linhas.append(dist)
                node_meta['rows'].append({
                    'row':ci,
                    'parents':dict(zip(parents,comb)),
                    'confidence':None,
                    'source':'LLM_FULLDIST_FALLBACK_NONORDINAL',
                    'distribution':dist,
                })

            node_meta['rn_parameters']=None

        else:
            # 1) Uma triagem de confiança para todas as configurações do nó.
            conf_file=detalhes_dir/f'{re.sub("[^A-Za-z0-9_-]","_",node)}_confidence.json'

            if conf_file.is_file():
                cdata=json.loads(conf_file.read_text(encoding='utf-8'))
                conf={int(k):float(v) for k,v in cdata['row_confidence'].items()}
                influencia=cdata['parent_influence']
                print('  [CHECKPOINT CONFIANÇA] reutilizado.',flush=True)
            else:
                pconf=format_prompt_confianca_no(nodes_info,node,parents,pcombs)
                conf,influencia,rawc=query_llm_confianca(
                    pconf,
                    parents,
                    len(pcombs),
                    system_prompt=system_prompt,
                    context={
                        'persona_id':etiqueta,
                        'rep':repeticao,
                        'node':node,
                        'pais':parents,
                    },
                )
                cdata={
                    'node':node,
                    'parents':parents,
                    'row_confidence':{str(k):v for k,v in conf.items()},
                    'parent_influence':influencia,
                    'raw':rawc,
                    'prompt':pconf,
                    'limiar':LIMIAR_CONFIANCA,
                    'utc':datetime.now(timezone.utc).isoformat(),
                }
                _salvar_json_atomico(conf_file,cdata)

            selecionadas=[i for i in range(len(pcombs)) if conf[i]>=LIMIAR_CONFIANCA]
            print(
                f'  [CONFIANÇA] {len(selecionadas)}/{len(pcombs)} linhas '
                f'>= {LIMIAR_CONFIANCA:.2f}: {selecionadas}',
                flush=True,
            )

            # 2) FullDist somente nas linhas de alta confiança.
            ancoras=[]
            dist_anchors={}

            for ci in selecionadas:
                comb=pcombs[ci]
                arq=detalhes_dir/f'{re.sub("[^A-Za-z0-9_-]","_",node)}_anchor_{ci}.json'

                if arq.is_file():
                    reg=json.loads(arq.read_text(encoding='utf-8'))
                    dist=reg['distribution']
                    print(f'    [CHECKPOINT ÂNCORA {ci}]',dist,flush=True)
                else:
                    prompt=format_epk_prompt_full_distribution(nodes_info,node,parents,comb)
                    dist,raw=query_llm_full_distribution(
                        prompt,
                        len(states),
                        system_prompt=system_prompt,
                        context={
                            'persona_id':etiqueta,
                            'rep':repeticao,
                            'node':node,
                            'pais':dict(zip(parents,comb)),
                            'states':states,
                            'fase':'anchor_fullDist',
                            'confidence':conf[ci],
                        },
                    )
                    reg={
                        'node':node,
                        'row':ci,
                        'parents':dict(zip(parents,comb)),
                        'confidence':conf[ci],
                        'source':'LLM_FULLDIST',
                        'distribution':dist,
                        'raw':raw,
                        'prompt':prompt,
                        'utc':datetime.now(timezone.utc).isoformat(),
                    }
                    _salvar_json_atomico(arq,reg)

                dist_anchors[ci]=dist
                ancoras.append({
                    'row':ci,
                    'comb':comb,
                    'confidence':conf[ci],
                    'distribution':dist,
                })

            # 3) Ajustar RN somente às âncoras; gold nunca entra aqui.
            params=ajustar_ranked_nodes(
                nodes_info,
                node,
                parents,
                pcombs,
                ancoras,
                influencia,
            )
            params.update({
                'parent_names':parents,
                'parent_influence_llm':influencia,
                'anchor_rows':selecionadas,
                'threshold':LIMIAR_CONFIANCA,
            })

            _salvar_json_atomico(
                detalhes_dir/f'{re.sub("[^A-Za-z0-9_-]","_",node)}_rn_fit.json',
                params,
            )

            print(
                '  [RN]',
                {k:params[k] for k in ('weights','directions','sigma','loss','fit_status','n_anchors')},
                flush=True,
            )

            # 4) CPT final: âncoras permanecem LLM; demais linhas vêm do RN.
            linhas=[]
            for ci,comb in enumerate(pcombs):
                if ci in dist_anchors:
                    dist=dist_anchors[ci]
                    source='LLM_FULLDIST'
                else:
                    mu=_mu_linha(
                        nodes_info,
                        parents,
                        comb,
                        params['weights'],
                        params['directions'],
                    )
                    dist=_rn_probs(mu,params['sigma'],len(states)).tolist()
                    source='RANKED_NODES'

                linhas.append(dist)
                node_meta['rows'].append({
                    'row':ci,
                    'parents':dict(zip(parents,comb)),
                    'confidence':conf[ci],
                    'source':source,
                    'distribution':dist,
                })

            node_meta['rn_parameters']=params
            node_meta['confidence_summary']={
                'min':float(min(conf.values())),
                'max':float(max(conf.values())),
                'mean':float(np.mean(list(conf.values()))),
                'n_high_confidence':len(selecionadas),
                'n_ranked_nodes':len(pcombs)-len(selecionadas),
            }

        kwargs=dict(
            variable=node,
            variable_card=len(states),
            values=np.asarray(linhas,dtype=float).T,
            state_names={node:states,**{p:nodes_info[p]['states'] for p in parents}},
        )
        if parents:
            kwargs.update(
                evidence=parents,
                evidence_card=[len(nodes_info[p]['states']) for p in parents],
            )

        cpds.append(TabularCPD(**kwargs))
        metadados['nodes'][node]=node_meta
        print(f'[NÓ CONCLUÍDO] {node} | {len(cpds)}/{len(model.nodes())} CPTs',flush=True)

    metadados['duration_seconds']=round(time.perf_counter()-inicio,3)
    metadados['call_counter_total_session']=LLMAPI.call_counter
    return cpds,metadados


def estimar_rede_confidence_rn(nodes_info,edges,persona=None,repeticao=0,checkpoint_dir=None,verbose=True):
    rede=construir_rede_vazia(nodes_info,edges)
    cpds,meta=gerar_cpts_confidence_rn(
        rede,
        nodes_info,
        persona=persona,
        repeticao=repeticao,
        checkpoint_dir=checkpoint_dir,
        verbose=verbose,
    )
    rede.add_cpds(*cpds)
    assert rede.check_model(), 'pgmpy: estrutura/CPT híbrida inválida'
    return rede,meta


In [ ]:
def export_result(model, nodes_info, output_path='resultado_rede_hibrida.json', hybrid_metadata=None):
    data={
        'nodes':nodes_info,
        'edges':[{'source':src,'target':dst} for src,dst in model.edges()],
        'cpds':{},
        'generation_method':METODO_HIBRIDO,
    }

    if hybrid_metadata is not None:
        data['hybrid_metadata']=hybrid_metadata

    for cpd in model.get_cpds():
        data['cpds'][cpd.variable]={
            'variables':list(cpd.variables),
            'state_names':cpd.state_names,
            'values':cpd.get_values().tolist(),
        }

    with open(output_path,'w',encoding='utf-8') as f:
        json.dump(data,f,ensure_ascii=False,indent=2)

    print(f'\n[INFO] Resultado híbrido exportado para: {output_path}')


## 6. Consulta inicial: o modelo afirma conhecer a rede?

In [ ]:
# Pré-etapa diagnóstica: uma consulta por modelo/rede/sessão, sem persona e sem qualquer CPT gold.
FAMILIARIDADE_REDE = None
FAMILIARIDADE_VERSAO = 'v1_apenas_nome_sem_topologia_antes_das_CPTs'

def _prompt_familiaridade_rede(nome):
    sistema = (
        'You are participating in a scientific study of Bayesian network parameter elicitation. '
        'Report uncertainty accurately. Do not pretend to remember a benchmark, graph or CPT '
        'when you only recognize its general topic. Do not invent numerical probabilities. '
        'You have no external tools in this query.'
    )
    pergunta = (
        f'Before being shown any graph, node descriptions, states, data, or probability tables: '
        f'were you ALREADY familiar with the SPECIFIC named Bayesian network benchmark "{nome}"? '
        'Distinguish clearly: (a) knowledge of its general domain, (b) recognition of this exact '
        'benchmark/network by name, (c) prior knowledge of its original nodes and edges, and '
        '(d) prior knowledge of its original numerical CPTs. Answer yes, no, or uncertain '
        'for each aspect and briefly explain what you know, or say you do not know. '
        'Do not guess CPT numbers or claim that the generic domain is the named benchmark.'
    )
    return sistema, pergunta

def consultar_familiaridade_rede():
    global FAMILIARIDADE_REDE
    caminho=DIR_EXECUCAO/'familiaridade_rede.json'
    texto=DIR_EXECUCAO/'familiaridade_rede.log'
    original_hash=hashlib.sha256(CAMINHO_JSON.read_bytes()).hexdigest()
    if SOMENTE_RECOMBINAR or not PERGUNTAR_FAMILIARIDADE_REDE:
        if caminho.is_file():
            anterior=json.loads(caminho.read_text(encoding='utf-8'))
            if (anterior.get('modelo')==HF_MODEL_ID and
                anterior.get('sha256_json_rede')==original_hash and
                anterior.get('nome_rede')==NOME_REDE_BNREP):
                FAMILIARIDADE_REDE={**anterior,'registro_reutilizado':True}
                print('[FAMILIARIDADE] Registro anterior compatível reutilizado:', caminho,flush=True)
                return FAMILIARIDADE_REDE
        motivo=('somente_recombinar' if SOMENTE_RECOMBINAR else 'desativada_explicitamente')
        FAMILIARIDADE_REDE={'status':'nao_consultada','motivo':motivo,
                           'nome_rede':NOME_REDE_BNREP,'modelo':HF_MODEL_ID,
                           'sha256_json_rede':original_hash,'resposta_completa':None,
                           'autorrelato_nao_comprovacao':True}
        print('[FAMILIARIDADE] Consulta não executada:', motivo,flush=True)
        return FAMILIARIDADE_REDE
    sistema,pergunta=_prompt_familiaridade_rede(NOME_REDE_BNREP)
    print('\n'+'='*78, '\n[PRIMEIRA CONSULTA] Conhecimento prévio declarado sobre a rede:',NOME_REDE_BNREP,flush=True)
    print('[FAMILIARIDADE] Apenas o nome da rede; sem nós, descrições, arestas, CPTs e sem persona.',flush=True)
    anterior_log=LLMAPI.log_path
    LLMAPI.log_path=DIR_EXECUCAO/'familiaridade_rede_chamadas.jsonl'
    inicio=time.perf_counter()
    try:
        resposta=LLMAPI.call(sistema,pergunta,temperature=0.0,max_retries=2,
                            max_new_tokens=384,
                            context={'metodo':'FamiliaridadeRede','persona_id':'sem_persona',
                                     'rep':0,'nome_rede':NOME_REDE_BNREP})
    finally:
        LLMAPI.log_path=anterior_log
    registro={
        'status':'respondida', 'versao_pergunta':FAMILIARIDADE_VERSAO,
        'utc':datetime.now(timezone.utc).isoformat(),'id_sessao':ID_SESSAO,
        'modelo':HF_MODEL_ID,'revisao_solicitada':HF_MODEL_REVISION,
        'qwen35_sem_thinking':QWEN35_SEM_THINKING,
        'nome_rede':NOME_REDE_BNREP,'sha256_json_rede':original_hash,
        'sem_persona':True,'antes_da_elicitacao':True,'gold_bnrep_ainda_nao_consultada':True,
        'pergunta_sistema':sistema,'pergunta_usuario':pergunta,'resposta_completa':resposta,
        'duracao_segundos':round(time.perf_counter()-inicio,3),
        'temperatura':0.0,'tokens_maximos':384,
        'caminho_chamadas':str(DIR_EXECUCAO/'familiaridade_rede_chamadas.jsonl'),
        'autorrelato_nao_comprovacao':True,
        'resposta_nao_injetada_nos_prompts_das_CPTs':True
    }
    # Salvar antes de começar qualquer método: válido inclusive se o Pod cair após a pergunta.
    _salvar_json_atomico(caminho,registro)
    texto.write_text('[PERGUNTA INICIAL - FAMILIARIDADE]\n'+pergunta+
                     '\n\n[RESPOSTA COMPLETA]\n'+resposta+'\n',encoding='utf-8')
    FAMILIARIDADE_REDE=registro
    print('[FAMILIARIDADE] Resposta completa (autorrelato; NÃO comprova memorização):\n'+resposta,flush=True)
    print('[FAMILIARIDADE] Arquivos:',caminho,'|',texto,flush=True)
    print('[FAMILIARIDADE] Resposta guardada apenas como diagnóstico; não altera as CPTs.',flush=True)
    return registro

## 7. Execução real — controle + 3 personas

Esta célula cria **quatro redes completas e independentes**:

- controle sem persona;
- engenheiro estrutural;
- sismologista;
- analista de risco.

Em cada condição, o protocolo é `Confidence screening -> FullDist nas âncoras -> Ranked Nodes nas demais linhas`. O bnRep continua fora desta etapa.


In [ ]:
import contextlib, shutil, sys, traceback
from datetime import datetime,timezone

SCHEMA_GERACAO = 'qwen35_confidence_ranked_nodes_v1'
DIR_GERACAO = None


def sha256_arquivo(p):
    return hashlib.sha256(Path(p).read_bytes()).hexdigest()


def _salvar_json_atomico(path, content):
    path=Path(path)
    path.parent.mkdir(parents=True,exist_ok=True)
    temp=path.with_suffix(path.suffix+'.tmp')
    temp.write_text(
        json.dumps(content,ensure_ascii=False,indent=2,default=str),
        encoding='utf-8',
    )
    temp.replace(path)


class Tee:
    def __init__(self,path):
        self.path=Path(path)
        self.path.parent.mkdir(parents=True,exist_ok=True)
        self.file=None
        self.console=None

    def __enter__(self):
        self.console=sys.stdout
        self.file=self.path.open('a',encoding='utf-8',buffering=1)
        sys.stdout=self
        return self

    def __exit__(self,*args):
        sys.stdout=self.console
        self.file.close()

    def write(self,text):
        self.console.write(text)
        self.file.write(text)

    def flush(self):
        self.console.flush()
        self.file.flush()


def validar_entradas_geracao():
    global PERSONAS_CONFIG,PERSONAS_ATIVAS,PERSONAS_HASH
    global REPETICOES,SOMENTE_RECOMBINAR,REALIZAR_INFERENCIA,DIR_EXECUCAO,DIR_GERACAO

    if not CAMINHO_JSON.is_file() or not CAMINHO_PERSONAS.is_file():
        raise FileNotFoundError(
            'Envie manualmente os dois JSONs para /content e confira CAMINHO_REDE_MANUAL e CAMINHO_PERSONAS_MANUAL.'
        )

    nos,arestas=load_bn_spec(str(CAMINHO_JSON))
    PERSONAS_CONFIG=carregar_personas_json(CAMINHO_PERSONAS,nos)
    PERSONAS_ATIVAS=PERSONAS_CONFIG['personas']
    PERSONAS_HASH=PERSONAS_CONFIG['hash']
    REPETICOES=PERSONAS_CONFIG['repeticoes']

    if PERSONAS_CONFIG['somente_recombinar']:
        raise ValueError('Desative calibracao.somente_recombinar: este notebook gera CPTs.')

    SOMENTE_RECOMBINAR=False
    REALIZAR_INFERENCIA=True

    config_material={
        'schema':SCHEMA_GERACAO,
        'json_rede_sha256':sha256_arquivo(CAMINHO_JSON),
        'json_personas_sha256':sha256_arquivo(CAMINHO_PERSONAS),
        'persona_ids':[p['id'] for p in PERSONAS_ATIVAS],
        'model':HF_MODEL_ID,
        'revision':HF_MODEL_REVISION,
        '4bits':USAR_4BITS,
        'seed':SEED,
        'temperature':TEMPERATURE,
        'repeticoes':REPETICOES,
        'metodo':METODO_HIBRIDO,
        'limiar_confianca':LIMIAR_CONFIANCA,
        'rn_sigma_default':RN_SIGMA_DEFAULT,
        'rn_sigma_bounds':[RN_SIGMA_MIN,RN_SIGMA_MAX],
        'rn_regularizacao_pesos':RN_REGULARIZACAO_PESOS,
        'qwen35_sem_thinking':QWEN35_SEM_THINKING,
    }

    cid=hashlib.sha256(
        json.dumps(config_material,sort_keys=True).encode()
    ).hexdigest()[:16]

    DIR_GERACAO=DIR_ESTUDO/'geracoes'/cid
    DIR_GERACAO.mkdir(parents=True,exist_ok=True)
    DIR_EXECUCAO=DIR_GERACAO

    for nome,origem in [
        ('rede_original.json',CAMINHO_JSON),
        ('personas_original.json',CAMINHO_PERSONAS),
    ]:
        destino=DIR_GERACAO/nome
        if destino.exists() and sha256_arquivo(destino)!=sha256_arquivo(origem):
            raise RuntimeError(f'Entrada congelada difere: {destino}')
        if not destino.exists():
            shutil.copy2(origem,destino)

    plano=[]
    condicoes=[('controle',None)]+[(p['id'],p) for p in PERSONAS_ATIVAS]

    for rep in range(REPETICOES):
        for cond,persona in condicoes:
            plano.append({
                'condicao':cond,
                'persona_id':persona['id'] if persona else None,
                'repeticao':rep+1,
                'metodo':METODO_HIBRIDO,
                'cpt_file':f'cpts/{cond}_{METODO_HIBRIDO}_r{rep+1}.json',
                'metadata_file':f'detalhes/{cond}_{METODO_HIBRIDO}_r{rep+1}.json',
                'console_log':f'logs/{cond}_{METODO_HIBRIDO}_r{rep+1}.log',
                'status':'pendente',
            })

    manifesto=DIR_GERACAO/'manifesto_geracao.json'

    if manifesto.is_file():
        antigo=json.loads(manifesto.read_text(encoding='utf-8'))
        if antigo.get('configuracao')!=config_material:
            raise RuntimeError('Manifesto existente incompatível.')

        anteriores={
            (x['condicao'],x['repeticao']):x
            for x in antigo['execucoes']
        }
        for x in plano:
            chave=(x['condicao'],x['repeticao'])
            if chave in anteriores:
                x.update(anteriores[chave])
        criado=antigo.get('criado_em_utc')
    else:
        criado=datetime.now(timezone.utc).isoformat()

    dados={
        'schema':SCHEMA_GERACAO,
        'status':'em_andamento',
        'criado_em_utc':criado,
        'atualizado_em_utc':datetime.now(timezone.utc).isoformat(),
        'configuracao':config_material,
        'nome_rede_bnrep_apenas_identificador':NOME_REDE_BNREP,
        'sem_consulta_ao_bnrep':True,
        'sem_dados_observados':True,
        'modelo':HF_MODEL_ID,
        'json_rede':'rede_original.json',
        'json_personas':'personas_original.json',
        'pesos_personas_nao_aplicados_na_geracao':True,
        'personas_executadas':[p['id'] for p in PERSONAS_ATIVAS],
        'familiaridade_arquivo':'familiaridade_rede.json' if PERGUNTAR_FAMILIARIDADE_REDE else None,
        'execucoes':plano,
    }
    _salvar_json_atomico(manifesto,dados)

    # Estimativa de custo antes de conhecermos quantas linhas superarão o limiar.
    g=nx.DiGraph()
    g.add_nodes_from(nos)
    g.add_edges_from(arestas)

    roots=[n for n in nos if g.in_degree(n)==0]
    rn_nodes=[
        n for n in nos
        if g.in_degree(n)>0 and _rn_elegivel(nos,n,list(g.predecessors(n)))
    ]

    fallback_rows=sum(
        int(np.prod([len(nos[p]['states']) for p in g.predecessors(n)]))
        for n in nos
        if g.in_degree(n)>0 and n not in rn_nodes
    )

    minimo_por_cond=len(roots)+len(rn_nodes)+fallback_rows

    full_rows=sum(
        int(np.prod([len(nos[p]['states']) for p in g.predecessors(n)]))
        if g.in_degree(n)>0 else 1
        for n in nos
    )

    print('[VALIDADO]',len(nos),'nós;',len(arestas),'arestas;',full_rows,'linhas CPT.',flush=True)
    print('[CONDIÇÕES] Controle + 3 personas:',[p['id'] for p in PERSONAS_ATIVAS],flush=True)
    print('[PROTOCOLO] Sem SepState. FullDist somente em raízes/âncoras/fallback não ordinal.',flush=True)
    print(
        '[CHAMADAS BASE POR CONDIÇÃO antes das âncoras]',
        minimo_por_cond,
        '= raízes',len(roots),
        '+ triagens',len(rn_nodes),
        '+ fallback não ordinal',fallback_rows,
        flush=True,
    )
    print('[OBS] Cada linha que superar o limiar adiciona 1 chamada FullDist.',flush=True)
    print('[RESULTADOS]',DIR_GERACAO,flush=True)

    return nos,arestas,dados,manifesto


def gerar_experimento():
    global FAMILIARIDADE_REDE

    nos,arestas,manifest,arq_manifest=validar_entradas_geracao()

    if not EXECUTAR_GERACAO:
        print('[DESATIVADO] Apenas conferência dos JSONs.')
        return manifest

    pendentes=[
        x for x in manifest['execucoes']
        if x['status']!='concluido' or not (DIR_GERACAO/x['cpt_file']).is_file()
    ]

    familia=DIR_GERACAO/'familiaridade_rede.json'
    precisa_familiaridade=PERGUNTAR_FAMILIARIDADE_REDE and not familia.is_file()

    if not pendentes and not precisa_familiaridade:
        manifest['status']='concluido'
        _salvar_json_atomico(arq_manifest,manifest)
        print('[CONCLUÍDO] Todas as redes já existem.')
        return manifest

    print('[MODELO] Carregando uma única vez. bnRep NÃO será consultado.',flush=True)
    carregar_llama()

    import torch
    manifest['versao_transformers']=__import__('transformers').__version__
    manifest['modelo_commit_carregado']=getattr(_MODEL.config,'_commit_hash',None)
    manifest['gpu']=torch.cuda.get_device_name(0)
    _salvar_json_atomico(arq_manifest,manifest)

    if PERGUNTAR_FAMILIARIDADE_REDE:
        if familia.is_file():
            anterior=json.loads(familia.read_text(encoding='utf-8'))
            if (
                anterior.get('modelo')!=HF_MODEL_ID
                or anterior.get('sha256_json_rede')!=sha256_arquivo(CAMINHO_JSON)
            ):
                raise RuntimeError('Familiaridade anterior refere-se a rede/modelo diferentes.')
            FAMILIARIDADE_REDE={**anterior,'registro_reutilizado':True}
            print('[FAMILIARIDADE] Registro anterior reutilizado.',flush=True)
        else:
            consultar_familiaridade_rede()

    for item in manifest['execucoes']:
        destino=DIR_GERACAO/item['cpt_file']
        meta_path=DIR_GERACAO/item['metadata_file']

        if item['status']=='concluido' and destino.is_file():
            print('[CONDIÇÃO EXISTENTE]',item['condicao'],'rep',item['repeticao'])
            continue

        cond=item['condicao']
        persona=next((p for p in PERSONAS_ATIVAS if p['id']==cond),None)
        rep=item['repeticao']-1
        log=DIR_GERACAO/item['console_log']

        try:
            with Tee(log):
                print('\n'+'='*78,flush=True)
                print('[CONDIÇÃO]',cond,'|',METODO_HIBRIDO,'| rep',rep+1,flush=True)
                print(
                    '[SEM GOLD] Estrutura + descrições + persona. '
                    'Confiança decide quais linhas recebem FullDist.',
                    flush=True,
                )

                inicio=time.perf_counter()

                rede,meta=estimar_rede_confidence_rn(
                    nos,
                    arestas,
                    persona=persona,
                    repeticao=rep,
                    checkpoint_dir=DIR_ESTUDO/'checkpoints_confidence_rn',
                    verbose=True,
                )

                destino.parent.mkdir(parents=True,exist_ok=True)
                meta_path.parent.mkdir(parents=True,exist_ok=True)

                export_result(rede,nos,str(destino),hybrid_metadata=meta)
                _salvar_json_atomico(meta_path,meta)

                tempo=round(time.perf_counter()-inicio,3)
                fingerprint=meta['fingerprint']
                audit_dir=DIR_ESTUDO/'auditoria_confidence_rn'

                for origem,nome in [
                    (audit_dir/f'consultas_{fingerprint}.jsonl','chamadas'),
                    (audit_dir/f'parser_{fingerprint}.jsonl','parser'),
                ]:
                    alvo=DIR_GERACAO/'logs'/f'{cond}_{METODO_HIBRIDO}_r{rep+1}_{nome}.jsonl'
                    alvo.parent.mkdir(parents=True,exist_ok=True)
                    if origem.is_file():
                        shutil.copy2(origem,alvo)

                n_llm=sum(
                    1
                    for nm in meta['nodes'].values()
                    for row in nm.get('rows',[])
                    if str(row.get('source','')).startswith('LLM_')
                )
                n_rn=sum(
                    1
                    for nm in meta['nodes'].values()
                    for row in nm.get('rows',[])
                    if row.get('source')=='RANKED_NODES'
                )

                item.update({
                    'status':'concluido',
                    'duracao_segundos':tempo,
                    'sha256_cpt':sha256_arquivo(destino),
                    'fingerprint_checkpoint':fingerprint,
                    'linhas_llm':n_llm,
                    'linhas_ranked_nodes':n_rn,
                    'concluido_em_utc':datetime.now(timezone.utc).isoformat(),
                })

                print(
                    '[SALVO]',
                    destino,
                    '| linhas LLM=',n_llm,
                    '| linhas RN=',n_rn,
                    '|',tempo,'s',
                    flush=True,
                )

            manifest['atualizado_em_utc']=datetime.now(timezone.utc).isoformat()
            _salvar_json_atomico(arq_manifest,manifest)

        except BaseException as exc:
            item['status']='interrompido'
            item['erro']=repr(exc)
            manifest['status']='interrompido'
            manifest['atualizado_em_utc']=datetime.now(timezone.utc).isoformat()
            _salvar_json_atomico(arq_manifest,manifest)
            print('[FALHA] Checkpoints preservados; execute novamente para retomar.',flush=True)
            raise

    manifest['status']='concluido'
    manifest['concluido_em_utc']=datetime.now(timezone.utc).isoformat()
    _salvar_json_atomico(arq_manifest,manifest)

    print('\n[GERAÇÃO FINALIZADA]',DIR_GERACAO,flush=True)
    return manifest


In [ ]:
MANIFESTO_GERACAO = gerar_experimento()

## 8. Inspecionar as saídas

A tabela abaixo mostra as quatro condições. `linhas_llm` informa quantas linhas finais vieram diretamente de FullDist; `linhas_ranked_nodes` informa quantas foram completadas pelo Ranked Nodes. Os arquivos `detalhes/*.json` preservam confiança, âncoras, pesos, direções, sigma e origem de cada linha.


In [ ]:
from IPython.display import display
import pandas as pd

if MANIFESTO_GERACAO:
    cols=[
        'condicao',
        'metodo',
        'repeticao',
        'status',
        'cpt_file',
        'metadata_file',
        'linhas_llm',
        'linhas_ranked_nodes',
        'duracao_segundos',
    ]
    df=pd.DataFrame(MANIFESTO_GERACAO['execucoes'])
    display(df[[c for c in cols if c in df.columns]])
    print('PASTA_GERACAO:',DIR_GERACAO)


## 9. Resultados no Colab

Os **resultados, manifestos, logs e checkpoints** ficam em:

`/content/Qwen_BN_Confidence_RN/`

Eles permanecem disponíveis enquanto o runtime do Colab estiver ativo.

Como esta versão **não usa Google Drive**, uma reinicialização completa do runtime apaga os arquivos locais. Ao final, execute a célula abaixo para gerar um `.zip` e faça o download manualmente pelo painel **Arquivos** do Colab.


In [ ]:
# ============================================================
# EMPACOTAR RESULTADOS PARA DOWNLOAD MANUAL
# ============================================================
import shutil
from pathlib import Path

if DIR_GERACAO is None or not Path(DIR_GERACAO).is_dir():
    raise RuntimeError('Nenhuma pasta de geração encontrada. Execute primeiro o experimento.')

ARQUIVO_ZIP = shutil.make_archive(
    '/content/resultados_qwen35_9b_confidence_ranked_nodes',
    'zip',
    root_dir=DIR_GERACAO
)

print('ZIP criado:', ARQUIVO_ZIP)
print('Baixe-o manualmente pelo painel Arquivos do Google Colab.')
